# Notebook 1: Data Loading & Preprocessing
### EEEM073 — AI and Sustainability | University of Surrey | 2025–2026
This notebook loads and preprocesses the EA EDM Storm Overflow Annual Returns (2023–2024) and English Indices of Multiple Deprivation (IMD) 2019.
**Inputs:** Raw EA EDM Excel files + IMD CSV
**Outputs:** edm_processed.csv, edm_site_imd.csv, edm_yoy.csv

## Step 1: Imports and Path Configuration

In [ ]:

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# --- Paths ---
RAW = "../data/raw/"
PROCESSED = "../data/processed/"

# --- File paths ---
EDM_2023 = os.path.join(RAW, "EDM_2023_Storm_Overflow_Annual_Return",
            "EDM 2023 Storm Overflow Annual Return - all water and sewerage companies.xlsx")
EDM_2024 = os.path.join(RAW, "EDM_2024_Storm_Overflow_Annual_Return",
            "EDM 2024 Storm Overflow Annual Return - all water and sewerage companies.xlsx")
IMD_PATH = os.path.join(RAW, "imd_2019_file7.csv")

# --- Verify all files exist ---
for label, path in [("EDM 2023", EDM_2023), ("EDM 2024", EDM_2024), ("IMD", IMD_PATH)]:
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"{label}: {status}")

EDM 2023: FOUND
EDM 2024: FOUND
IMD: FOUND


## Step 2: Inspect Raw Excel Structure
Before bulk loading, we inspect one sheet to identify the correct header row. EDM files contain metadata rows above the actual column headers.

In [ ]:


xl_2023 = pd.ExcelFile(EDM_2023)
xl_2024 = pd.ExcelFile(EDM_2024)

print("=== 2023 Sheets ===")
for sheet in xl_2023.sheet_names:
    print(f"  {sheet}")

print("\n=== 2024 Sheets ===")
for sheet in xl_2024.sheet_names:
    print(f"  {sheet}")

=== 2023 Sheets ===
  Anglian Water 2023
  DCWW 2023
  Northumbrian Water 2023
  Severn Trent 2023
  South West Water 2023
  Southern Water 2023
  Thames Water 2023
  United Utilities 2023
  Wessex Water 2023
  Yorkshire Water 2023

=== 2024 Sheets ===
  Anglian Water 2024
  DCWW 2024
  Northumbrian Water 2024
  Severn Trent 2024
  South West Water 2024
  Southern Water 2024
  Thames Water 2024
  United Utilities 2024
  Wessex Water 2024
  Yorkshire Water 2024


## Step 3: Load All Water Companies — Both Years
Each Excel file contains 10 company sheets. We load all sheets into a single combined DataFrame.

In [ ]:

# Headers are on row index 1, data starts row index 2
def load_edm_file(filepath, year):
    """
    Load all sheets from an EDM annual return Excel file.
    Each sheet = one water and sewerage company.
    
    Args:
        filepath: path to the Excel file
        year: integer year (2023 or 2024)
    Returns:
        Combined DataFrame of all companies for that year
    """
    xl = pd.ExcelFile(filepath)
    all_sheets = []
    
    for sheet in xl.sheet_names:
        df = pd.read_excel(filepath, sheet_name=sheet, header=1)
        df['year'] = year
        df['source_sheet'] = sheet
        all_sheets.append(df)
        print(f"  Loaded: {sheet} — {len(df)} rows")
    
    combined = pd.concat(all_sheets, ignore_index=True)
    return combined

print("Loading 2023...")
edm_2023 = load_edm_file(EDM_2023, 2023)

print("\nLoading 2024...")
edm_2024 = load_edm_file(EDM_2024, 2024)

print(f"\n2023 total rows: {len(edm_2023)}")
print(f"2024 total rows: {len(edm_2024)}")

Loading 2023...
  Loaded: Anglian Water 2023 — 1563 rows
  Loaded: DCWW 2023 — 127 rows
  Loaded: Northumbrian Water 2023 — 1565 rows
  Loaded: Severn Trent 2023 — 2472 rows
  Loaded: South West Water 2023 — 1374 rows
  Loaded: Southern Water 2023 — 977 rows
  Loaded: Thames Water 2023 — 698 rows
  Loaded: United Utilities 2023 — 2264 rows
  Loaded: Wessex Water 2023 — 1295 rows
  Loaded: Yorkshire Water 2023 — 2195 rows

Loading 2024...
  Loaded: Anglian Water 2024 — 1444 rows
  Loaded: DCWW 2024 — 128 rows
  Loaded: Northumbrian Water 2024 — 1560 rows
  Loaded: Severn Trent 2024 — 2459 rows
  Loaded: South West Water 2024 — 1370 rows
  Loaded: Southern Water 2024 — 978 rows
  Loaded: Thames Water 2024 — 590 rows
  Loaded: United Utilities 2024 — 2270 rows
  Loaded: Wessex Water 2024 — 1295 rows
  Loaded: Yorkshire Water 2024 — 2191 rows

2023 total rows: 14530
2024 total rows: 14285


## Step 4: Combine 2023 and 2024

In [ ]:


edm_raw = pd.concat([edm_2023, edm_2024], ignore_index=True)

print(f"Combined dataset: {edm_raw.shape[0]} rows x {edm_raw.shape[1]} columns")
print(f"\nColumns available:")
for i, col in enumerate(edm_raw.columns):
    print(f"  [{i:02d}] {col}")

Combined dataset: 28815 rows x 42 columns

Columns available:
  [00] Water Company Name
  [01] Site Name
(EA Consents Database)
  [02] Site Name
(WaSC operational)
[optional]
  [03] EA Permit Reference
(EA Consents Database)
  [04] WaSC Supplementary Permit Ref.
[optional]
  [05] Activity Reference on Permit
  [06] Storm Discharge Asset Type
  [07] Outlet Discharge NGR
(EA Consents Database)
  [08] WFD Waterbody ID (Cycle 2)
(discharge outlet)
  [09] WFD Waterbody Catchment Name (Cycle 2)
(discharge outlet)
  [10] Receiving Water / Environment (common name)
(EA Consents Database)
  [11] Shellfish Water (only populate for storm overflow with a Shellfish Water EDM requirement)
  [12] Bathing Water (only populate for storm overflow with a Bathing Water EDM requirement)
  [13] Treatment Method
(over & above Storm Tank settlement / screening)
  [14] Initial EDM Commission Date
  [15] Total Duration (hrs) all spills prior to processing through 12-24h count method
  [16] Counted spills using 

## Step 5: Select and Rename Key Columns
Verbose multi-line column names are mapped to clean, code-friendly equivalents.

In [ ]:


# Map original verbose column names to clean names
column_map = {
    'Water Company Name': 'company',
    'Site Name\n(EA Consents Database)': 'site_name',
    'EA Permit Reference\n(EA Consents Database)': 'permit_ref',
    'Storm Discharge Asset Type': 'asset_type',
    'Outlet Discharge NGR\n(EA Consents Database)': 'grid_ref',
    'WFD Waterbody ID (Cycle 2)\n(discharge outlet)': 'waterbody_id',
    'WFD Waterbody Catchment Name (Cycle 2)\n(discharge outlet)': 'catchment_name',
    'Receiving Water / Environment (common name)\n(EA Consents Database)': 'receiving_water',
    'Bathing Water (only populate for storm overflow with a Bathing Water EDM requirement)': 'bathing_water',
    'Total Duration (hrs) all spills prior to processing through 12-24h count method': 'total_spill_hours',
    'Counted spills using 12-24h count method': 'spill_count',
    'Long-term average spill count': 'spill_count_avg',
    'No. full years EDM data\n(years)': 'edm_years',
    'EDM Operation -\n% of reporting period EDM operational': 'edm_pct_operational',
    'Unique ID': 'unique_id',
    'year': 'year',
    'source_sheet': 'source_sheet'
}

# Keep only columns that exist in our data
valid_cols = {k: v for k, v in column_map.items() if k in edm_raw.columns}
edm = edm_raw[list(valid_cols.keys())].rename(columns=valid_cols)

print(f"Selected {len(edm.columns)} columns")
print(f"Shape: {edm.shape}")
print(f"\nFirst 3 rows:")
edm.head(3)

Selected 17 columns
Shape: (28815, 17)

First 3 rows:


,company,site_name,permit_ref,asset_type,grid_ref,waterbody_id,catchment_name,receiving_water,bathing_water,total_spill_hours,spill_count,spill_count_avg,edm_years,edm_pct_operational,unique_id,year,source_sheet
0,Anglian Water,ABTHORPE TERMINAL PUMPING STATION,AW1NF947,Storm discharge at pumping station,SP6419046470,GB105033038260,Tove (US Greens Norton),THE RIVER TOVE,N/a,0.0,0.0,0.4,4 (installed 2019),100.0,AnW0001,2023,Anglian Water 2023
1,Anglian Water,SCHOOL LANE SPS ABY,ANNNF2581,Storm discharge at pumping station,TF4142078410,GB105029061750,Woldgrift Drain,Un-Named Dyke to Great Eau,N/a,0.0,0.0,NaN,<1 yr (installed 2023),100.0,AnW0002,2023,Anglian Water 2023
2,Anglian Water,AKELEY (EX STW) PS,AWCNF10279,Storm discharge at pumping station,SP7106038010,GB105033037960,Leckhampstead Brook,Leckhampstead Brook,N/a,1698.0,104.0,49.5,3 (installed 2020),100.0,AnW0003,2023,Anglian Water 2023


## Step 6: Initial Data Quality Assessment
We inspect missing value patterns before making any cleaning decisions. Understanding why values are missing informs appropriate handling strategies.

In [ ]:


print("=== DATA QUALITY REPORT ===\n")
print(f"Total records: {len(edm)}")
print(f"Unique overflow sites (unique_id): {edm['unique_id'].nunique()}")
print(f"Companies: {sorted(edm['company'].dropna().unique())}")
print(f"Years: {edm['year'].unique()}")

print(f"\n=== MISSING VALUES ===")
missing = edm.isnull().sum()
missing_pct = (missing / len(edm) * 100).round(1)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)
print(missing_df.to_string())

print(f"\n=== KEY COLUMN STATS ===")
print(edm[['spill_count', 'total_spill_hours', 'edm_pct_operational']].describe().round(2))

=== DATA QUALITY REPORT ===

Total records: 28815
Unique overflow sites (unique_id): 26501
Companies: ['Anglian Water', 'Dwr Cymru Welsh Water', 'Northumbrian Water', 'Severn Trent Water', 'South West Water', 'Southern Water', 'Thames Water', 'United Utilities', 'Wessex Water', 'Yorkshire Water']
Years: [2023 2024]

=== MISSING VALUES ===
                     missing_count  missing_pct
bathing_water                26255         91.1
catchment_name               15098         52.4
total_spill_hours            14784         51.3
edm_years                    14504         50.3
waterbody_id                 14294         49.6
spill_count_avg               1220          4.2
spill_count                    602          2.1
edm_pct_operational            320          1.1
site_name                      282          1.0
receiving_water                227          0.8
permit_ref                     148          0.5
unique_id                      127          0.4
grid_ref                         1 

## Step 7: Investigate High Missingness
Three columns exceed 50% missing. We investigate whether these are structurally missing or data quality gaps.

In [ ]:

# Missing data in EDM files is meaningful - not random
# 1. bathing_water is 91% missing - expected, only coastal overflows need this
print("=== BATHING WATER (91% missing) ===")
print("Non-null count:", edm['bathing_water'].notna().sum())
print("This column only applies to coastal/bathing water overflows")
print("Decision: DROP - not relevant to inland sewage pollution analysis\n")

# 2. total_spill_hours - 51% missing
print("=== TOTAL SPILL HOURS (51% missing) ===")
# Check if missing hours correlates with zero spills
hours_missing_spills = edm[edm['total_spill_hours'].isna()]['spill_count'].describe()
print("Spill count stats where hours are missing:")
print(hours_missing_spills.round(2))

# 3. catchment_name - 52% missing
print("\n=== CATCHMENT NAME (52% missing) ===")
print("Missing by company:")
print(edm.groupby('company')['catchment_name'].apply(
    lambda x: f"{x.isna().sum()} / {len(x)} ({x.isna().mean()*100:.0f}%)"
))

# 4. edm_pct_operational - tells us monitor reliability
print("\n=== EDM OPERATIONAL % ===")
print("Monitors below 90% operational (unreliable data):")
unreliable = edm[edm['edm_pct_operational'] < 90]
print(f"  Count: {len(unreliable)} ({len(unreliable)/len(edm)*100:.1f}%)")

=== BATHING WATER (91% missing) ===
Non-null count: 2560
This column only applies to coastal/bathing water overflows
Decision: DROP - not relevant to inland sewage pollution analysis

=== TOTAL SPILL HOURS (51% missing) ===
Spill count stats where hours are missing:
count    14182.00
mean        31.76
std         38.08
min          0.00
25%          4.00
50%         19.00
75%         46.00
max        366.00
Name: spill_count, dtype: float64

=== CATCHMENT NAME (52% missing) ===
Missing by company:
company
Anglian Water            1451 / 3007 (48%)
Dwr Cymru Welsh Water      156 / 255 (61%)
Northumbrian Water       1560 / 3125 (50%)
Severn Trent Water       2459 / 4931 (50%)
South West Water         1908 / 2744 (70%)
Southern Water            978 / 1955 (50%)
Thames Water              590 / 1288 (46%)
United Utilities         2510 / 4534 (55%)
Wessex Water             1295 / 2590 (50%)
Yorkshire Water          2191 / 4386 (50%)
Name: catchment_name, dtype: str

=== EDM OPERATIONAL % ===

## Step 8: Data Cleaning — Documented Decisions
Every cleaning decision is logged with a justification. Removing unreliable monitor records would bias the dataset toward well-resourced areas — we flag instead of remove.

In [ ]:


edm_clean = edm.copy()
cleaning_log = []

# --- Decision 1: Drop bathing_water ---
# 91.1% missing - only applies to coastal overflows, not relevant to our analysis
edm_clean = edm_clean.drop(columns=['bathing_water'])
cleaning_log.append({
    'column': 'bathing_water',
    'action': 'dropped',
    'reason': '91.1% missing - only applies to coastal overflows, irrelevant to inland analysis',
    'rows_affected': 0
})

# --- Decision 2: Drop rows where unique_id is missing ---
# unique_id is our primary key for joining with IMD - cannot proceed without it
before = len(edm_clean)
edm_clean = edm_clean.dropna(subset=['unique_id'])
after = len(edm_clean)
cleaning_log.append({
    'column': 'unique_id',
    'action': f'dropped {before-after} rows',
    'reason': 'Primary key required for IMD join - cannot identify site without it',
    'rows_affected': before - after
})

# --- Decision 3: Flag unreliable monitors, don't drop ---
# Monitors <90% operational may undercount spills - flag rather than remove
# Removing would bias toward well-resourced areas (fairness concern)
edm_clean['monitor_reliable'] = edm_clean['edm_pct_operational'] >= 90
unreliable_count = (~edm_clean['monitor_reliable']).sum()
cleaning_log.append({
    'column': 'edm_pct_operational',
    'action': f'flagged {unreliable_count} unreliable monitors',
    'reason': 'Removing unreliable monitors would bias toward well-resourced areas - flag instead',
    'rows_affected': unreliable_count
})

# --- Decision 4: Fill missing spill_count with 0 where monitor IS reliable ---
# A reliable monitor with no recorded spills = genuinely 0 spills
mask_reliable_no_spill = (edm_clean['monitor_reliable']) & (edm_clean['spill_count'].isna())
edm_clean.loc[mask_reliable_no_spill, 'spill_count'] = 0
filled_count = mask_reliable_no_spill.sum()
cleaning_log.append({
    'column': 'spill_count',
    'action': f'filled {filled_count} nulls with 0',
    'reason': 'Reliable monitor + null spill count = genuinely no spills recorded',
    'rows_affected': filled_count
})

# --- Decision 5: Fill missing total_spill_hours with 0 where spill_count is 0 ---
mask_no_spills = (edm_clean['spill_count'] == 0) & (edm_clean['total_spill_hours'].isna())
edm_clean.loc[mask_no_spills, 'total_spill_hours'] = 0
cleaning_log.append({
    'column': 'total_spill_hours',
    'action': f'filled {mask_no_spills.sum()} nulls with 0',
    'reason': 'Zero spills recorded = zero duration, not missing data',
    'rows_affected': int(mask_no_spills.sum())
})

# --- Decision 6: Fill catchment_name nulls with 'Unknown' ---
# Do not drop - losing half the dataset would severely bias analysis
edm_clean['catchment_name'] = edm_clean['catchment_name'].fillna('Unknown')
cleaning_log.append({
    'column': 'catchment_name',
    'action': 'filled nulls with Unknown',
    'reason': 'Dropping 52% of data would critically bias the dataset - retain with flag',
    'rows_affected': int(edm['catchment_name'].isna().sum())
})

# --- Decision 7: Fill remaining edm_pct_operational nulls with median ---
median_op = edm_clean['edm_pct_operational'].median()
edm_clean['edm_pct_operational'] = edm_clean['edm_pct_operational'].fillna(median_op)
cleaning_log.append({
    'column': 'edm_pct_operational',
    'action': f'filled nulls with median ({median_op:.1f}%)',
    'reason': 'Small number of nulls (1.1%) - median imputation preserves distribution',
    'rows_affected': 320
})

# --- Print cleaning summary ---
print("=== CLEANING LOG ===\n")
for entry in cleaning_log:
    print(f"Column: {entry['column']}")
    print(f"  Action: {entry['action']}")
    print(f"  Reason: {entry['reason']}")
    print()

print(f"=== FINAL SHAPE ===")
print(f"Before cleaning: {len(edm)} rows")
print(f"After cleaning:  {len(edm_clean)} rows")
print(f"Remaining nulls:\n{edm_clean.isnull().sum()[edm_clean.isnull().sum() > 0]}")

=== CLEANING LOG ===

Column: bathing_water
  Action: dropped
  Reason: 91.1% missing - only applies to coastal overflows, irrelevant to inland analysis

Column: unique_id
  Action: dropped 127 rows
  Reason: Primary key required for IMD join - cannot identify site without it

Column: edm_pct_operational
  Action: flagged 4406 unreliable monitors
  Reason: Removing unreliable monitors would bias toward well-resourced areas - flag instead

Column: spill_count
  Action: filled 0 nulls with 0
  Reason: Reliable monitor + null spill count = genuinely no spills recorded

Column: total_spill_hours
  Action: filled 1772 nulls with 0
  Reason: Zero spills recorded = zero duration, not missing data

Column: catchment_name
  Action: filled nulls with Unknown
  Reason: Dropping 52% of data would critically bias the dataset - retain with flag

Column: edm_pct_operational
  Action: filled nulls with median (99.9%)
  Reason: Small number of nulls (1.1%) - median imputation preserves distribution

==

## Step 9: Parse OS National Grid References to Coordinates
We convert NGR strings to decimal latitude/longitude for spatial IMD joining.

In [ ]:

# We need coordinates to join with IMD deprivation data
# Grid refs format: e.g. SU4933095090

import re

def parse_ngr(ngr):
    """
    Parse OS National Grid Reference to approximate 
    decimal latitude/longitude.
    Returns (lat, lon) tuple or (None, None) if invalid.
    
    NGR format: 2-letter prefix + easting digits + northing digits
    e.g. SU4933095090 -> prefix SU, easting 49330, northing 95090
    """
    if pd.isna(ngr) or not isinstance(ngr, str):
        return None, None
    
    # OS grid letter offsets
    grid_letters = {
        'SV': (0, 0), 'SW': (1, 0), 'SX': (2, 0), 'SY': (3, 0), 'SZ': (4, 0),
        'TV': (5, 0), 'SR': (1, 1), 'SS': (2, 1), 'ST': (3, 1), 'SU': (4, 1),
        'TQ': (5, 1), 'TR': (6, 1), 'SM': (1, 2), 'SN': (2, 2), 'SO': (3, 2),
        'SP': (4, 2), 'TL': (5, 2), 'TM': (6, 2), 'SH': (2, 3), 'SJ': (3, 3),
        'SK': (4, 3), 'TF': (5, 3), 'TG': (6, 3), 'SC': (2, 4), 'SD': (3, 4),
        'SE': (4, 4), 'TA': (5, 4), 'NX': (2, 5), 'NY': (3, 5), 'NZ': (4, 5),
        'NT': (3, 6), 'NU': (4, 6), 'NS': (2, 6), 'NH': (2, 7), 'NJ': (3, 7),
        'NK': (4, 7), 'NG': (1, 7), 'NM': (2, 8), 'NN': (3, 8), 'NO': (4, 8),
        'NB': (1, 9), 'NC': (2, 9), 'ND': (3, 9), 'NA': (0, 9), 'HW': (1, 11),
        'HX': (2, 11), 'HY': (3, 11), 'HZ': (4, 11), 'HU': (3, 10), 'HT': (2, 10)
    }
    
    ngr = ngr.strip().upper().replace(' ', '')
    match = re.match(r'^([A-Z]{2})(\d+)$', ngr)
    if not match:
        return None, None
    
    prefix = match.group(1)
    digits = match.group(2)
    
    if prefix not in grid_letters:
        return None, None
    
    half = len(digits) // 2
    easting_local = int(digits[:half])
    northing_local = int(digits[half:])
    
    ex, ey = grid_letters[prefix]
    easting = ex * 100000 + easting_local
    northing = ey * 100000 + northing_local
    
    # Convert OSGB36 to approximate WGS84 lat/lon
    # Simple approximation sufficient for regional analysis
    lat = 49.766 + (northing / 1000000) * 5.5
    lon = -7.557 + (easting / 1000000) * 9.9
    
    return round(lat, 5), round(lon, 5)

# Apply to dataset
print("Parsing grid references...")
edm_clean[['lat', 'lon']] = edm_clean['grid_ref'].apply(
    lambda x: pd.Series(parse_ngr(x))
)

valid_coords = edm_clean['lat'].notna().sum()
print(f"Successfully parsed: {valid_coords} / {len(edm_clean)} grid references")
print(f"Coverage: {valid_coords/len(edm_clean)*100:.1f}%")
print(f"\nSample coordinates:")
print(edm_clean[['site_name', 'grid_ref', 'lat', 'lon']].dropna().head(5).to_string())

Parsing grid references...
Successfully parsed: 28632 / 28688 grid references
Coverage: 99.8%

Sample coordinates:
                           site_name      grid_ref       lat      lon
0  ABTHORPE TERMINAL PUMPING STATION  SP6419046470  51.12158 -2.96152
1                SCHOOL LANE SPS ABY  TF4142078410  51.84725 -2.19694
2                 AKELEY (EX STW) PS  SP7106038010  51.07505 -2.89351
3  ALDBOROUGH WATER RECYCLING CENTRE  TG1858033590  51.60074 -1.43306
4                      ALDEBURGH STW  TM4735054700  51.16685 -1.14824


## Step 10: Load IMD Deprivation Data

In [ ]:

# Join to overflow sites using Local Authority District

# Load IMD file
imd = pd.read_csv(IMD_PATH)
print(f"IMD loaded: {imd.shape}")
print(f"Columns: {list(imd.columns[:8])}")
print(f"\nFirst row sample:")
print(imd.iloc[0, :8])

IMD loaded: (32844, 57)
Columns: ['LSOA code (2011)', 'LSOA name (2011)', 'Local Authority District code (2019)', 'Local Authority District name (2019)', 'Index of Multiple Deprivation (IMD) Score', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)', 'Income Score (rate)']

First row sample:
LSOA code (2011)                                                                                E01000001
LSOA name (2011)                                                                      City of London 001A
Local Authority District code (2019)                                                            E09000001
Local Authority District name (2019)                                                       City of London
Index of Multiple Deprivation (IMD) Score                                                           6.208
Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)     

## Step 11: Aggregate IMD to Local Authority District Level
IMD operates at LSOA level. We aggregate to LAD level to match overflow site coordinate resolution.

In [ ]:

# overflow sites have grid references but not LSOA codes
# Strategy: aggregate IMD to Local Authority District level
# then join via company-to-LAD mapping

# Rename IMD columns to clean names
imd_clean = imd[[
    'Local Authority District code (2019)',
    'Local Authority District name (2019)',
    'Index of Multiple Deprivation (IMD) Score',
    'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)',
    'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)',
    'Income Score (rate)',
    'Health Deprivation and Disability Score',
    'Crime Score'
]].rename(columns={
    'Local Authority District code (2019)': 'lad_code',
    'Local Authority District name (2019)': 'lad_name',
    'Index of Multiple Deprivation (IMD) Score': 'imd_score',
    'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)': 'imd_rank',
    'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)': 'imd_decile',
    'Income Score (rate)': 'income_score',
    'Health Deprivation and Disability Score': 'health_score',
    'Crime Score': 'crime_score'
})

# Aggregate to LAD level — mean IMD score per Local Authority
imd_lad = imd_clean.groupby(['lad_code', 'lad_name']).agg(
    imd_score_mean=('imd_score', 'mean'),
    imd_score_max=('imd_score', 'max'),
    imd_decile_mean=('imd_decile', 'mean'),
    income_score_mean=('income_score', 'mean'),
    health_score_mean=('health_score', 'mean'),
    crime_score_mean=('crime_score', 'mean'),
    n_lsoas=('lad_code', 'count')
).reset_index()

# Create IMD quintile at LAD level (1=most deprived, 5=least deprived)
imd_lad['imd_quintile'] = pd.qcut(
    imd_lad['imd_score_mean'],
    q=5,
    labels=[5, 4, 3, 2, 1]  # reversed so 1=most deprived matches IMD convention
).astype(int)

print(f"IMD aggregated to {len(imd_lad)} Local Authority Districts")
print(f"\nIMD quintile distribution:")
print(imd_lad['imd_quintile'].value_counts().sort_index())
print(f"\nSample:")
print(imd_lad.head(3).to_string())

IMD aggregated to 317 Local Authority Districts

IMD quintile distribution:
imd_quintile
1    64
2    63
3    63
4    63
5    64
Name: count, dtype: int64

Sample:
    lad_code              lad_name  imd_score_mean  imd_score_max  imd_decile_mean  income_score_mean  health_score_mean  crime_score_mean  n_lsoas  imd_quintile
0  E06000001            Hartlepool       34.853121         74.719         3.689655           0.226241           0.874190          0.582345       58             1
1  E06000002         Middlesbrough       40.443116         78.009         3.383721           0.251326           1.192070          0.680523       86             1
2  E06000003  Redcar and Cleveland       29.841330         76.969         4.340909           0.185534           0.807034          0.152227       88             1


## Step 12: Map Water Companies to Local Authorities

In [ ]:


# Company to LAD name keyword mapping
# Based on EA water company service areas
company_region_keywords = {
    'Thames Water': ['London', 'Surrey', 'Berkshire', 'Oxfordshire', 
                     'Buckinghamshire', 'Hertfordshire', 'Essex', 'Kent'],
    'Anglian Water': ['Norfolk', 'Suffolk', 'Essex', 'Cambridgeshire',
                      'Lincolnshire', 'Northamptonshire', 'Bedfordshire'],
    'Severn Trent Water': ['Birmingham', 'Coventry', 'Derby', 'Nottingham',
                           'Leicester', 'Staffordshire', 'Worcestershire'],
    'Yorkshire Water': ['Leeds', 'Sheffield', 'Bradford', 'Hull',
                        'York', 'Barnsley', 'Doncaster', 'Rotherham'],
    'United Utilities': ['Manchester', 'Liverpool', 'Cumbria', 'Cheshire',
                         'Lancashire', 'Merseyside'],
    'Northumbrian Water': ['Newcastle', 'Sunderland', 'Durham', 'Middlesbrough',
                           'Gateshead', 'Northumberland'],
    'Southern Water': ['Hampshire', 'West Sussex', 'East Sussex', 'Kent', 'Isle of Wight'],
    'South West Water': ['Devon', 'Cornwall', 'Somerset', 'Dorset'],
    'Wessex Water': ['Bristol', 'Bath', 'Somerset', 'Dorset', 'Wiltshire'],
    'Dwr Cymru Welsh Water': ['Cardiff', 'Swansea', 'Newport', 'Pembrokeshire']
}

def get_company_imd(company_name, imd_df, keywords_map):
    """
    Get mean IMD score for a water company's service area
    by matching LAD names to known regional keywords.
    """
    if company_name not in keywords_map:
        return None
    keywords = keywords_map[company_name]
    mask = imd_df['lad_name'].apply(
        lambda x: any(kw.lower() in x.lower() for kw in keywords)
    )
    subset = imd_df[mask]
    if len(subset) == 0:
        return None
    return subset['imd_score_mean'].mean()

# Build company-level IMD lookup
company_imd = {}
for company in edm_clean['company'].dropna().unique():
    score = get_company_imd(company, imd_lad, company_region_keywords)
    company_imd[company] = score
    print(f"  {company}: IMD score = {score:.3f}" if score else f"  {company}: not matched")

# Map to main dataset
edm_clean['company_imd_score'] = edm_clean['company'].map(company_imd)

  Anglian Water: IMD score = 16.612
  Northumbrian Water: IMD score = 28.212
  Severn Trent Water: IMD score = 21.675
  South West Water: IMD score = 17.265
  Southern Water: IMD score = 16.842
  Thames Water: IMD score = 9.990
  United Utilities: IMD score = 26.765
  Wessex Water: IMD score = 17.184
  Yorkshire Water: IMD score = 26.342
  Dwr Cymru Welsh Water: not matched


## Step 13: Feature Engineering
Seven new features derived from raw columns, each motivated by sewage overflow domain knowledge.

In [ ]:

# Creating new informative features from raw columns
# features our ML models will learn from


# --- 1. Spill severity score ---
# Combines frequency and duration into single severity metric
# A site that spills 100 times for 1hr is different to 1 spill for 100hrs
edm_clean['spill_severity'] = (
    edm_clean['spill_count'] * edm_clean['total_spill_hours']
).fillna(0)

# --- 2. Average spill duration per event ---
edm_clean['avg_spill_duration_hrs'] = np.where(
    edm_clean['spill_count'] > 0,
    edm_clean['total_spill_hours'] / edm_clean['spill_count'],
    0
)

# --- 3. Spill rate vs long-term average ---
# How much worse/better is this year vs historical average?
# > 1 means more spills than average, < 1 means fewer
edm_clean['spill_vs_average'] = np.where(
    edm_clean['spill_count_avg'] > 0,
    edm_clean['spill_count'] / edm_clean['spill_count_avg'],
    np.nan
)

# --- 4. High spill flag ---
# Binary: is this site in the top 25% worst spillers this year?
threshold_75 = edm_clean['spill_count'].quantile(0.75)
edm_clean['high_spiller'] = (edm_clean['spill_count'] >= threshold_75).astype(int)

# --- 5. Year-over-year change (only possible for sites in both years) ---
# Pivot to get 2023 and 2024 side by side per site
yoy = edm_clean.pivot_table(
    index='unique_id',
    columns='year',
    values='spill_count',
    aggfunc='first'
).reset_index()
yoy.columns = ['unique_id', 'spill_2023', 'spill_2024']
yoy['yoy_change'] = yoy['spill_2024'] - yoy['spill_2023']
yoy['yoy_pct_change'] = ((yoy['spill_2024'] - yoy['spill_2023']) / 
                          yoy['spill_2023'].replace(0, np.nan) * 100).round(2)

# Join back
edm_clean = edm_clean.merge(yoy[['unique_id', 'yoy_change', 'yoy_pct_change']], 
                              on='unique_id', how='left')

# --- 6. Monitor reliability category ---
edm_clean['reliability_cat'] = pd.cut(
    edm_clean['edm_pct_operational'],
    bins=[0, 75, 90, 100],
    labels=['poor', 'acceptable', 'good']
)

print("=== ENGINEERED FEATURES SUMMARY ===\n")
new_features = ['spill_severity', 'avg_spill_duration_hrs', 
                'spill_vs_average', 'high_spiller', 
                'yoy_change', 'yoy_pct_change', 'reliability_cat']
print(edm_clean[new_features].describe().round(2))
print(f"\nHigh spillers: {edm_clean['high_spiller'].sum()} sites ({edm_clean['high_spiller'].mean()*100:.1f}%)")

=== ENGINEERED FEATURES SUMMARY ===

       spill_severity  avg_spill_duration_hrs  spill_vs_average  high_spiller  \
count        28688.00                16278.00          26119.00      28688.00   
mean         12626.59                    3.58              1.22          0.25   
std          61558.16                    4.91              0.95          0.43   
min              0.00                    0.00              0.00          0.00   
25%              0.00                    0.00              0.71          0.00   
50%              0.00                    1.30              1.10          0.00   
75%            524.62                    5.22              1.52          0.00   
max        1871100.66                   24.32             25.85          1.00   

       yoy_change  yoy_pct_change  
count     4288.00         3702.00  
mean        -4.44           29.30  
std         20.16          272.81  
min       -202.00         -100.00  
25%        -12.00          -36.70  
50%         -1.00

## Step 14: Save Processed Dataset

In [ ]:


output_path = "../data/processed/edm_processed.csv"
edm_clean.to_csv(output_path, index=False)

# Also save IMD LAD aggregation
imd_output = "../data/processed/imd_lad.csv"
imd_lad.to_csv(imd_output, index=False)

# Also save YoY comparison
yoy_output = "../data/processed/edm_yoy.csv"
yoy.to_csv(yoy_output, index=False)

print("=== SAVED FILES ===")
print(f"Main dataset:    {output_path}")
print(f"  Shape: {edm_clean.shape}")
print(f"  Columns: {list(edm_clean.columns)}")
print(f"\nIMD LAD data:    {imd_output}")
print(f"  Shape: {imd_lad.shape}")
print(f"\nYoY comparison:  {yoy_output}")
print(f"  Shape: {yoy.shape}")

print(f"\n=== FINAL DATASET SUMMARY ===")
print(f"Total records:        {len(edm_clean)}")
print(f"Unique sites:         {edm_clean['unique_id'].nunique()}")
print(f"Companies:            {edm_clean['company'].nunique()}")
print(f"Years:                {sorted(edm_clean['year'].unique())}")
print(f"Features:             {edm_clean.shape[1]}")
print(f"Missing values:       {edm_clean.isnull().sum().sum()}")

=== SAVED FILES ===
Main dataset:    ../data/processed/edm_processed.csv
  Shape: (28688, 27)
  Columns: ['company', 'site_name', 'permit_ref', 'asset_type', 'grid_ref', 'waterbody_id', 'catchment_name', 'receiving_water', 'total_spill_hours', 'spill_count', 'spill_count_avg', 'edm_years', 'edm_pct_operational', 'unique_id', 'year', 'source_sheet', 'monitor_reliable', 'lat', 'lon', 'company_imd_score', 'spill_severity', 'avg_spill_duration_hrs', 'spill_vs_average', 'high_spiller', 'yoy_change', 'yoy_pct_change', 'reliability_cat']

IMD LAD data:    ../data/processed/imd_lad.csv
  Shape: (317, 10)

YoY comparison:  ../data/processed/edm_yoy.csv
  Shape: (25949, 5)

=== FINAL DATASET SUMMARY ===
Total records:        28688
Unique sites:         26501
Companies:            10
Years:                [np.int64(2023), np.int64(2024)]
Features:             27
Missing values:       109144


In [15]:
print(edm_clean.isnull().sum()[edm_clean.isnull().sum() > 0].sort_values(ascending=False))

yoy_pct_change            24986
yoy_change                24400
edm_years                 14504
waterbody_id              14294
total_spill_hours         13005
avg_spill_duration_hrs    12410
spill_vs_average           2569
spill_count_avg            1218
spill_count                 595
site_name                   282
reliability_cat             275
receiving_water             227
permit_ref                  138
company_imd_score           128
lon                          56
lat                          56
grid_ref                      1
dtype: int64


In [ ]:

# Final null handling — small remaining gaps


# spill_count — fill remaining nulls with 0
# These are unreliable monitors with no recorded data
edm_clean['spill_count'] = edm_clean['spill_count'].fillna(0)
edm_clean['total_spill_hours'] = edm_clean['total_spill_hours'].fillna(0)
edm_clean['avg_spill_duration_hrs'] = edm_clean['avg_spill_duration_hrs'].fillna(0)

# company_imd_score — fill with national median
national_median_imd = edm_clean['company_imd_score'].median()
edm_clean['company_imd_score'] = edm_clean['company_imd_score'].fillna(national_median_imd)

# lat/lon — 56 sites with unparseable grid refs, flag them
edm_clean['coords_available'] = edm_clean['lat'].notna().astype(int)

# yoy columns — fill with 0, meaning no change detected
# (structurally missing = site only in one year = no change measurable)
edm_clean['yoy_change'] = edm_clean['yoy_change'].fillna(0)
edm_clean['yoy_pct_change'] = edm_clean['yoy_pct_change'].fillna(0)

# reliability_cat — fill remaining with 'unknown'
edm_clean['reliability_cat'] = edm_clean['reliability_cat'].cat.add_categories('unknown')
edm_clean['reliability_cat'] = edm_clean['reliability_cat'].fillna('unknown')

# Verify
remaining = edm_clean.isnull().sum()
remaining = remaining[remaining > 0]
print("=== REMAINING NULLS AFTER FINAL CLEANING ===")
if len(remaining) == 0:
    print("None — dataset is fully clean")
else:
    print(remaining)

# Overwrite saved file with final clean version
edm_clean.to_csv("../data/processed/edm_processed.csv", index=False)
print(f"\nFinal dataset saved: {edm_clean.shape}")
print(f"Ready for EDA")

=== REMAINING NULLS AFTER FINAL CLEANING ===
site_name             282
permit_ref            138
grid_ref                1
waterbody_id        14294
receiving_water       227
spill_count_avg      1218
edm_years           14504
lat                    56
lon                    56
spill_vs_average     2569
dtype: int64

Final dataset saved: (28688, 28)
Ready for EDA


## Notebook 1 Summary
| Item | Value |
|---|---|
| Total records | 28,688 |
| Unique sites | 26,501 |
| Companies | 10 |
| Years | 2023, 2024 |
| Output files | edm_processed.csv, edm_site_imd.csv, edm_yoy.csv |

**Proceed to:** 2_eda.ipynb

---